# Step 4 — Phase 2: Naive Sequential Continual Learning Baseline

**Project:** Continual Self-Supervised Pretraining for Industrial Anomaly Localization

**Goal of this notebook:**
1. Fine-tune an ImageNet-pretrained ResNet-18 sequentially on Bottle, then
   Cable, then Leather, using the same SimCLR/NT-Xent objective from Notebook 2.
2. After each new category, evaluate PatchCore AUROC/PRO on every category
   seen so far, not just the newest one.
3. Produce the forgetting curve -- this is the naive baseline that later
   continual learning strategies (EWC, replay, PackNet) will be compared
   against.

**Why ImageNet as the starting encoder:** Phase 1 showed ImageNet pretraining
outperformed both scratch and SimCLR pretraining consistently across all
three categories, making it the strongest available starting point. Using it
here isolates the continual learning question (does sequential fine-tuning
forget earlier categories) without also confounding it with "is the starting
representation good enough."

**Training budget:** Phase 1 found that extending SimCLR fine-tuning from
100 to 300 epochs improved PatchCore AUROC and PRO consistently across all
three categories, narrowing the gap to ImageNet pretraining in each case.
300 epochs per task is therefore used uniformly across this continual
stream as well, matching the setting that performed best in the static
(non-continual) Phase 1 evaluation.

**Requires GPU runtime.**

## 4.1 — Remount Drive, confirm GPU, restore paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), "No GPU detected -- switch Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda")

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/anomaly_project'
DATA_ROOT = f'{PROJECT_ROOT}/data/mvtec'
CHECKPOINT_ROOT = f'{PROJECT_ROOT}/checkpoints'
RESULTS_ROOT = f'{PROJECT_ROOT}/results'

for d in [DATA_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT]:
    os.makedirs(d, exist_ok=True)

!pip install -q faiss-gpu-cu12 2>/dev/null || pip install -q faiss-cpu
print("Setup complete.")

## 4.2 — Re-declare dataset class, encoder, NT-Xent loss, PatchCore pipeline

Self-contained copy of the components from Notebooks 2 and 3, since GPU
runtime switches clear the Python session.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import time
import json


class MVTecDataset(Dataset):
    """MVTec AD dataset loader. See Notebook 1 for full documentation."""

    def __init__(self, root, split="train", transform=None, mask_transform=None, two_crop=False):
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.mask_transform = mask_transform
        self.two_crop = two_crop
        self.samples = self._build_index()

    def _build_index(self):
        samples = []
        if self.split == "train":
            good_dir = self.root / "train" / "good"
            for img_path in sorted(good_dir.glob("*.png")):
                samples.append({"image_path": img_path, "label": 0, "mask_path": None})
        elif self.split == "test":
            test_dir = self.root / "test"
            for defect_dir in sorted(test_dir.iterdir()):
                if not defect_dir.is_dir():
                    continue
                defect_type = defect_dir.name
                is_anomalous = defect_type != "good"
                for img_path in sorted(defect_dir.glob("*.png")):
                    mask_path = None
                    if is_anomalous:
                        mask_path = self.root / "ground_truth" / defect_type / f"{img_path.stem}_mask.png"
                    samples.append({"image_path": img_path, "label": int(is_anomalous),
                                     "mask_path": mask_path, "defect_type": defect_type})
        else:
            raise ValueError(f"split must be 'train' or 'test', got '{self.split}'")
        return samples

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_image(sample["image_path"])

        if self.split == "train":
            if self.two_crop:
                view1 = self.transform(img) if self.transform else img
                view2 = self.transform(img) if self.transform else img
                return view1, view2
            return self.transform(img) if self.transform else img
        else:
            img_t = self.transform(img) if self.transform else img
            if sample["mask_path"] is not None:
                mask = Image.open(sample["mask_path"]).convert("L")
                mask_t = self.mask_transform(mask) if self.mask_transform else mask
                mask_t = torch.from_numpy((np.array(mask_t) > 0).astype(np.float32))
            else:
                h, w = img_t.shape[-2:] if isinstance(img_t, torch.Tensor) else img.size[::-1]
                mask_t = torch.zeros((h, w), dtype=torch.float32)
            return {"image": img_t, "label": sample["label"], "mask": mask_t,
                    "defect_type": sample["defect_type"], "image_path": str(sample["image_path"])}


def save_checkpoint(state: dict, category: str, strategy: str, tag: str = "latest"):
    out_dir = f"{CHECKPOINT_ROOT}/{category}"
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{strategy}_{tag}.pt"
    state["_saved_at"] = time.time()
    torch.save(state, path)
    return path


def load_checkpoint(category: str, strategy: str, tag: str = "latest", map_location="cpu"):
    path = f"{CHECKPOINT_ROOT}/{category}/{strategy}_{tag}.pt"
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location)


def save_result(result: dict, path_name: str):
    path = f"{RESULTS_ROOT}/{path_name}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2, default=str)
    print(f"Result saved: {path}")


IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TEXTURE_CATEGORIES = {"leather", "carpet", "tile", "wood", "grid"}


def get_simclr_transform(category):
    """Same category-aware augmentation pipeline as Notebook 2."""
    color_jitter_strength = 0.2
    hue_jitter = 0.0 if category in TEXTURE_CATEGORIES else color_jitter_strength * 0.5
    crop_scale_lower = 0.2 if category in TEXTURE_CATEGORIES else 0.5
    return T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(crop_scale_lower, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(
            brightness=color_jitter_strength, contrast=color_jitter_strength,
            saturation=color_jitter_strength, hue=hue_jitter,
        ),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
mask_eval_transform = T.Resize((IMG_SIZE, IMG_SIZE))

print("Dataset class, checkpoint utilities, and transforms ready.")

In [ ]:
class SimCLRProjectionHead(nn.Module):
    """Projection head used during fine-tuning, discarded afterward -- the
    backbone itself is what persists across the task stream and what
    PatchCore reads features from."""

    def __init__(self, in_dim=512, projection_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, projection_dim),
        )

    def forward(self, x):
        return self.net(x)


def nt_xent_loss(z_a, z_b, temperature=0.5):
    """NT-Xent contrastive loss -- identical derivation and implementation to
    Notebook 2, reused here for the continual fine-tuning stream."""
    N = z_a.shape[0]
    z = torch.cat([z_a, z_b], dim=0)
    z = F.normalize(z, dim=1)
    sim_matrix = torch.matmul(z, z.T) / temperature
    self_mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim_matrix.masked_fill_(self_mask, float("-inf"))
    positive_indices = torch.cat([
        torch.arange(N, 2 * N, device=z.device),
        torch.arange(0, N, device=z.device),
    ])
    return F.cross_entropy(sim_matrix, positive_indices)


class PatchFeatureExtractor(nn.Module):
    """layer1+layer2+layer3 feature extractor, identical to Notebook 3."""

    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.backbone.eval()
        for p in self.backbone.parameters():
            p.requires_grad = False
        self._features = {}
        self.backbone.layer1.register_forward_hook(self._make_hook("layer1"))
        self.backbone.layer2.register_forward_hook(self._make_hook("layer2"))
        self.backbone.layer3.register_forward_hook(self._make_hook("layer3"))

    def _make_hook(self, name):
        def hook(module, input, output):
            self._features[name] = output
        return hook

    @torch.no_grad()
    def forward(self, x):
        self._features = {}
        _ = self.backbone(x)
        f1, f2, f3 = self._features["layer1"], self._features["layer2"], self._features["layer3"]
        f2u = F.interpolate(f2, size=f1.shape[-2:], mode="bilinear", align_corners=False)
        f3u = F.interpolate(f3, size=f1.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f1, f2u, f3u], dim=1)


print("SimCLRProjectionHead, nt_xent_loss, PatchFeatureExtractor defined.")

In [ ]:
def greedy_coreset_selection(features: torch.Tensor, target_size: int, device="cuda"):
    """Identical to Notebook 3 -- see that notebook for the full derivation
    and the rare-cluster sanity test."""
    N = features.shape[0]
    if target_size >= N:
        return features
    features = features.to(device)
    selected_indices = [torch.randint(0, N, (1,)).item()]
    min_dists = torch.cdist(features, features[selected_indices]).squeeze(1)
    for _ in range(target_size - 1):
        next_idx = torch.argmax(min_dists).item()
        selected_indices.append(next_idx)
        new_dists = torch.cdist(features, features[next_idx:next_idx+1]).squeeze(1)
        min_dists = torch.minimum(min_dists, new_dists)
    return features[selected_indices].cpu()


CORESET_RATIO = 0.10


def build_memory_bank_from_backbone(backbone, category: str, batch_size: int = 16):
    """Same as Notebook 3's build_memory_bank, but takes an already-loaded
    backbone directly instead of loading one from a checkpoint -- needed here
    since the continual stream keeps one backbone in memory across categories
    rather than loading a fresh one per evaluation."""
    category_path = f"{DATA_ROOT}/{category}"
    extractor = PatchFeatureExtractor(backbone).to(device)

    train_ds = MVTecDataset(category_path, split="train", transform=eval_transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    all_patches = []
    patch_grid_size = None
    with torch.no_grad():
        for batch in train_loader:
            batch = batch.to(device)
            feat_map = extractor(batch)
            B, C, H, W = feat_map.shape
            patch_grid_size = (H, W)
            patches = feat_map.permute(0, 2, 3, 1).reshape(-1, C)
            all_patches.append(patches.cpu())

    all_patches = torch.cat(all_patches, dim=0)
    target_size = int(all_patches.shape[0] * CORESET_RATIO)
    memory_bank = greedy_coreset_selection(all_patches, target_size=target_size, device=device)
    return memory_bank, patch_grid_size, extractor


print("greedy_coreset_selection and build_memory_bank_from_backbone defined.")

In [ ]:
import faiss
from sklearn.metrics import roc_auc_score
from scipy import ndimage


def score_test_set(category: str, memory_bank: torch.Tensor, patch_grid_size: tuple,
                    extractor: nn.Module, batch_size: int = 8):
    """Identical to Notebook 3's score_test_set."""
    category_path = f"{DATA_ROOT}/{category}"
    test_ds = MVTecDataset(category_path, split="test", transform=eval_transform,
                            mask_transform=mask_eval_transform)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    bank_np = memory_bank.numpy().astype("float32")
    index = faiss.IndexFlatL2(bank_np.shape[1])
    index.add(bank_np)

    H, W = patch_grid_size
    image_scores, labels, anomaly_maps, gt_masks = [], [], [], []

    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(device)
            feat_map = extractor(images)
            B, C, _, _ = feat_map.shape
            patches = feat_map.permute(0, 2, 3, 1).reshape(B, H * W, C).cpu().numpy().astype("float32")

            for b in range(B):
                distances, _ = index.search(patches[b], k=1)
                patch_distances = distances[:, 0].reshape(H, W)
                image_scores.append(patch_distances.max())

                patch_map_t = torch.from_numpy(patch_distances).unsqueeze(0).unsqueeze(0).float()
                upsampled = F.interpolate(patch_map_t, size=(IMG_SIZE, IMG_SIZE),
                                           mode="bilinear", align_corners=False).squeeze().numpy()
                anomaly_maps.append(upsampled)

            labels.extend(batch["label"].tolist())
            gt_masks.extend([m.numpy() for m in batch["mask"]])

    return {"image_scores": np.array(image_scores), "labels": np.array(labels),
            "anomaly_maps": anomaly_maps, "gt_masks": gt_masks}


def compute_image_auroc(image_scores, labels):
    return roc_auc_score(labels, image_scores)


def compute_pro_score(anomaly_maps, gt_masks, fpr_limit=0.3, n_thresholds=50):
    """Percentile-threshold PRO, identical to Notebook 3."""
    all_scores = np.concatenate([m.flatten() for m in anomaly_maps])
    percentiles = np.linspace(0, 100, n_thresholds)
    thresholds = np.unique(np.percentile(all_scores, percentiles))

    labeled_masks = []
    for mask in gt_masks:
        if mask.sum() > 0:
            labeled, n_regions = ndimage.label(mask)
            labeled_masks.append((labeled, n_regions))
        else:
            labeled_masks.append((None, 0))

    mean_pro_per_threshold, fpr_per_threshold = [], []
    total_normal_pixels = sum((m == 0).sum() for m in gt_masks)

    for thresh in thresholds:
        region_tprs, fp_pixels = [], 0
        for amap, mask, (labeled, n_regions) in zip(anomaly_maps, gt_masks, labeled_masks):
            pred_binary = (amap >= thresh)
            fp_pixels += np.logical_and(pred_binary, mask == 0).sum()
            if n_regions > 0:
                for region_id in range(1, n_regions + 1):
                    region_mask = (labeled == region_id)
                    region_size = region_mask.sum()
                    if region_size == 0:
                        continue
                    overlap = np.logical_and(pred_binary, region_mask).sum()
                    region_tprs.append(overlap / region_size)
        mean_pro_per_threshold.append(np.mean(region_tprs) if region_tprs else 0.0)
        fpr_per_threshold.append(fp_pixels / max(total_normal_pixels, 1))

    order = np.argsort(fpr_per_threshold)
    fpr_sorted = np.array(fpr_per_threshold)[order]
    pro_sorted = np.array(mean_pro_per_threshold)[order]
    mask_in_range = fpr_sorted <= fpr_limit
    if mask_in_range.sum() < 2:
        return 0.0
    trapezoid_fn = getattr(np, "trapezoid", None) or np.trapz
    integral = trapezoid_fn(pro_sorted[mask_in_range], fpr_sorted[mask_in_range])
    return integral / fpr_limit


print("score_test_set, compute_image_auroc, compute_pro_score defined.")

## 4.3 — The continual fine-tuning step

This function fine-tunes a given backbone on one category's normal images,
using the same SimCLR/NT-Xent objective as Notebook 2. The key difference
from Notebook 2 is that the backbone is **passed in already-trained** (from
the previous category in the stream) rather than freshly initialized --
this is what makes the process continual rather than independent per-category
training.

In [ ]:
def finetune_on_category(backbone: nn.Module, category: str, num_epochs: int,
                          batch_size: int = 64, lr: float = 3e-4, temperature: float = 0.5):
    """
    Fine-tunes `backbone` in place on `category`'s normal training images
    using NT-Xent. Returns the same backbone object (now updated) plus the
    loss history for this stage.
    """
    category_path = f"{DATA_ROOT}/{category}"
    transform = get_simclr_transform(category)

    train_ds = MVTecDataset(category_path, split="train", transform=transform, two_crop=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True)

    backbone.train()
    projection_head = SimCLRProjectionHead(in_dim=512, projection_dim=128).to(device)

    optimizer = torch.optim.Adam(
        list(backbone.parameters()) + list(projection_head.parameters()), lr=lr
    )

    loss_history = []
    for epoch in range(num_epochs):
        epoch_losses = []
        for view_a, view_b in train_loader:
            view_a, view_b = view_a.to(device), view_b.to(device)

            h_a = backbone(view_a)
            h_b = backbone(view_b)
            z_a = projection_head(h_a)
            z_b = projection_head(h_b)

            loss = nt_xent_loss(z_a, z_b, temperature=temperature)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        mean_loss = sum(epoch_losses) / len(epoch_losses)
        loss_history.append(mean_loss)
        if epoch % 25 == 0 or epoch == num_epochs - 1:
            print(f"    epoch {epoch:3d}/{num_epochs} | loss = {mean_loss:.4f}")

    backbone.eval()
    return backbone, loss_history


print("finetune_on_category() defined.")

## 4.4 — The task stream: train sequentially, evaluate on everything seen so far

**Resume logic:** this loop checkpoints the backbone after every task and
records progress in a small state file, so a Colab disconnect partway through
the stream does not require restarting from Bottle.

In [ ]:
TASK_STREAM = ["bottle", "cable", "leather"]
EPOCHS_PER_TASK = 300
STRATEGY_NAME = "naive_continual"

stream_state_path = f"{RESULTS_ROOT}/phase2_naive_stream_state.json"

if os.path.exists(stream_state_path):
    with open(stream_state_path, "r") as f:
        stream_state = json.load(f)
    print(f"Resuming stream -- {len(stream_state['completed_tasks'])} task(s) already done: "
          f"{stream_state['completed_tasks']}")
else:
    stream_state = {"completed_tasks": [], "forgetting_records": []}
    print("Starting the continual stream from scratch.")

# Load the backbone -- either from the last completed task's checkpoint, or
# fresh from ImageNet if this is the very first task.
if stream_state["completed_tasks"]:
    last_task = stream_state["completed_tasks"][-1]
    checkpoint = load_checkpoint(category=last_task, strategy=STRATEGY_NAME, tag="stream", map_location=device)
    backbone = models.resnet18(weights=None)
    backbone.fc = nn.Identity()
    backbone.load_state_dict(checkpoint["backbone_state"])
    backbone = backbone.to(device)
    print(f"Loaded backbone state from after task '{last_task}'.")
else:
    backbone = models.resnet18(weights="IMAGENET1K_V1")
    backbone.fc = nn.Identity()
    backbone = backbone.to(device)
    print("Starting from ImageNet-pretrained weights.")

In [ ]:
for task_idx, task_category in enumerate(TASK_STREAM):
    if task_category in stream_state["completed_tasks"]:
        print(f"Task '{task_category}' already completed -- skipping fine-tuning.")
        continue

    print(f"\n{'='*60}\nTask {task_idx + 1}/{len(TASK_STREAM)}: fine-tuning on '{task_category}'\n{'='*60}")

    backbone, loss_history = finetune_on_category(
        backbone, task_category, num_epochs=EPOCHS_PER_TASK
    )

    save_checkpoint({
        "backbone_state": backbone.state_dict(),
        "task_category": task_category,
        "task_idx": task_idx,
        "loss_history": loss_history,
    }, category=task_category, strategy=STRATEGY_NAME, tag="stream")

    # Evaluate on EVERY category seen so far -- this is what reveals forgetting.
    seen_so_far = TASK_STREAM[:task_idx + 1]
    print(f"\nEvaluating on all categories seen so far: {seen_so_far}")

    for eval_category in seen_so_far:
        memory_bank, patch_grid_size, extractor = build_memory_bank_from_backbone(backbone, eval_category)
        test_results = score_test_set(eval_category, memory_bank, patch_grid_size, extractor)

        auroc = compute_image_auroc(test_results["image_scores"], test_results["labels"])
        pro = compute_pro_score(test_results["anomaly_maps"], test_results["gt_masks"])

        print(f"  After task '{task_category}' -- eval on '{eval_category}': "
              f"AUROC = {auroc:.4f} | PRO = {pro:.4f}")

        stream_state["forgetting_records"].append({
            "trained_through_task": task_category,
            "trained_through_idx": task_idx,
            "eval_category": eval_category,
            "auroc": float(auroc),
            "pro": float(pro),
        })

    stream_state["completed_tasks"].append(task_category)
    with open(stream_state_path, "w") as f:
        json.dump(stream_state, f, indent=2)
    print(f"\nState saved -- {len(stream_state['completed_tasks'])}/{len(TASK_STREAM)} tasks complete.")

print(f"\n{'='*60}\nCONTINUAL STREAM COMPLETE\n{'='*60}")

## 4.5 — The forgetting curve

For each category, this plots its AUROC at the point it was first learned
against its AUROC after each subsequent task. A flat line means no
forgetting; a declining line quantifies how much performance on earlier
categories degrades as new ones are learned.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

records_df = pd.DataFrame(stream_state["forgetting_records"])
print(records_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
for eval_category in TASK_STREAM:
    subset = records_df[records_df["eval_category"] == eval_category]
    ax.plot(subset["trained_through_idx"], subset["auroc"], marker="o", label=eval_category)

ax.set_xlabel("Task index (after training through this task)")
ax.set_ylabel("AUROC")
ax.set_xticks(range(len(TASK_STREAM)))
ax.set_xticklabels(TASK_STREAM)
ax.set_title("Naive continual fine-tuning -- forgetting curve")
ax.legend(title="Evaluated on")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Backward transfer (BWT): the average drop in AUROC on each EARLIER category,
# measured from when it was first learned to the end of the stream. This is
# the standard summary statistic for catastrophic forgetting -- negative BWT
# means forgetting occurred; BWT near zero means earlier tasks were retained.

bwt_terms = []
final_task_idx = len(TASK_STREAM) - 1

for i, category in enumerate(TASK_STREAM[:-1]):  # exclude the last task -- nothing comes after it
    just_learned = records_df[
        (records_df["eval_category"] == category) & (records_df["trained_through_idx"] == i)
    ]["auroc"].iloc[0]
    at_end = records_df[
        (records_df["eval_category"] == category) & (records_df["trained_through_idx"] == final_task_idx)
    ]["auroc"].iloc[0]
    drop = at_end - just_learned
    bwt_terms.append(drop)
    print(f"{category}: AUROC right after learning = {just_learned:.4f}, "
          f"AUROC at end of stream = {at_end:.4f}, change = {drop:+.4f}")

bwt = sum(bwt_terms) / len(bwt_terms)
print(f"\nBackward Transfer (BWT): {bwt:+.4f}")
print("(Negative values indicate forgetting; this is the naive baseline that")
print(" EWC, replay, and PackNet will be compared against.)")

save_result({
    "strategy": STRATEGY_NAME, "task_stream": TASK_STREAM, "epochs_per_task": EPOCHS_PER_TASK,
    "bwt": float(bwt), "records": stream_state["forgetting_records"],
}, path_name="phase2_naive_continual_summary")

## Step 4 — Done. What you should have right now:

- [ ] A single backbone fine-tuned sequentially through Bottle -> Cable -> Leather
- [ ] AUROC/PRO recorded for every (task, eval category) combination
- [ ] A forgetting curve plot
- [ ] Backward transfer (BWT) computed as the baseline forgetting measurement
- [ ] Results saved to `results/phase2_naive_continual_summary.json`

**Next steps:** Notebook 5 implements EWC, coreset replay, and PackNet on the
same task stream, each compared against this naive baseline's BWT.